# Portfolio-safe version

This notebook is a sanitized portfolio adaptation of the author's MSc Data Analytics project.
Environment-specific paths, cloud bucket names, notebook outputs, and exact patient examples
have been removed or generalized. The original methodology and core code structure are preserved.

**Data note:** the underlying TCIA imaging/clinical data are not redistributed in this repository.
Configure your own authorized/local dataset paths before running the notebook.


In [ ]:
# ============================================
# Colab script (clean): patient-level split + encodings + labels
#  - Filters clinical "NOTES/N/A" rows (keeps PatientID starting with STS_)
# ============================================
import re
from pathlib import Path
import numpy as np
import pandas as pd

# ---- INPUT PATHS ----
BASE_CSV = "data/03_Consolidated_radiomics_all_series.csv"
CLIN_XLSX = "data/INFOclinical_STS.xlsx"

# ---- OUTPUT DIR ----
OUTDIR = Path("data/processed_splits")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ---------------------------
# 1) Load radiomics (series)
# ---------------------------
rad = pd.read_csv(BASE_CSV)
rad.columns = [str(c).strip() for c in rad.columns]
assert "PatientID" in rad.columns and "StudyID" in rad.columns, \
    f"Expected columns 'PatientID' and 'StudyID'. Got: {rad.columns.tolist()}"

# ---------------------------
# 2) Extract StudyDate from StudyID
# ---------------------------
def extract_date_from_studyid(s):
    if pd.isna(s): return pd.NaT
    s = str(s)
    m = re.search(r'(\d{2})[-/](\d{2})[-/](\d{4})', s)  # dd-mm-yyyy or mm-dd-yyyy
    if m:
        d1 = pd.to_datetime("-".join(m.groups()), format="%d-%m-%Y", errors="coerce")
        if pd.isna(d1):
            d1 = pd.to_datetime("-".join(m.groups()), format="%m-%d-%Y", errors="coerce")
        return d1
    m2 = re.search(r'(\d{4})[-/](\d{2})[-/](\d{2})', s)  # yyyy-mm-dd
    if m2:
        return pd.to_datetime("-".join(m2.groups()), format="%Y-%m-%d", errors="coerce")
    return pd.NaT

rad["StudyDate"] = rad["StudyID"].apply(extract_date_from_studyid)

# ---------------------------
# 3) Patient study counts
# ---------------------------
counts = rad.groupby("PatientID")["StudyID"].nunique().reset_index(name="num_studies")
counts["num_rows"] = rad.groupby("PatientID")["StudyID"].size().values
counts = counts.sort_values("num_studies", ascending=False).reset_index(drop=True)

total_studies = int(rad["StudyID"].nunique())
targets = {
    "train": round(0.70 * total_studies),
    "val": round(0.15 * total_studies),
    "test": total_studies - (round(0.70 * total_studies) + round(0.15 * total_studies))
}

# ---------------------------
# 4) Patient-level split (~70/15/15 studies)
# ---------------------------
patients = counts.sample(frac=1.0, random_state=42).reset_index(drop=True)
remaining = targets.copy()
assignment = {}
priority = {"train": 2, "val": 1, "test": 0}

for _, row in patients.iterrows():
    best = max(remaining.keys(), key=lambda k: (remaining[k], priority[k]))
    assignment[row["PatientID"]] = best
    remaining[best] -= int(row["num_studies"])

def rebalance(assign, counts_df, targets_dict, max_iters=200):
    for _ in range(max_iters):
        achieved = {k:0 for k in targets_dict}
        for pid, sp in assign.items():
            ns = int(counts_df.loc[counts_df["PatientID"]==pid, "num_studies"].values[0])
            achieved[sp] += ns
        diffs = {k: targets_dict[k]-achieved[k] for k in targets_dict}
        worst_pos = max(diffs, key=lambda k: diffs[k])
        worst_neg = min(diffs, key=lambda k: diffs[k])
        if diffs[worst_pos] <= 0 or diffs[worst_neg] >= 0:
            break
        cand = counts_df[counts_df["PatientID"].isin([pid for pid, sp in assign.items() if sp==worst_neg])]
        if cand.empty: break
        cand = cand.copy()
        cand["effect"] = np.abs((diffs[worst_pos]-cand["num_studies"])) + np.abs((diffs[worst_neg]+cand["num_studies"]))
        move_pid = cand.sort_values("effect").iloc[0]["PatientID"]
        assign[move_pid] = worst_pos
    return assign

assignment = rebalance(assignment, counts, targets)

split_map = pd.DataFrame({"PatientID": list(assignment.keys()), "split": list(assignment.values())})
split_map = split_map.merge(counts, on="PatientID", how="left")
split_map.to_csv(OUTDIR / "patient_split_manifest.csv", index=False)

# ---------------------------
# 5) Apply split on study rows and order by date
# ---------------------------
rad2 = rad.merge(split_map[["PatientID","split"]], on="PatientID", how="left")
rad2 = rad2.sort_values(["PatientID","StudyDate","StudyID"]).reset_index(drop=True)

# ---------------------------
# 6) Load clinical workbook (two sheets) and merge
#     >>> FILTER NON-PATIENT ROWS (NOTES/N/A) <<<
# ---------------------------
xls = pd.ExcelFile(CLIN_XLSX)
clin_parts = []
for sh in xls.sheet_names:
    df_sh = pd.read_excel(xls, sheet_name=sh)
    df_sh.columns = [str(c).strip() for c in df_sh.columns]
    # Find PatientID column
    pid_col = None
    for c in df_sh.columns:
        if str(c).strip().lower() in ("patientid","patient_id","id","patient"):
            pid_col = c; break
    if pid_col is None:
        for c in df_sh.columns:
            if df_sh[c].astype(str).str.startswith("STS_").any():
                pid_col = c; break
    if pid_col is None:
        continue
    df_sh = df_sh.rename(columns={pid_col:"PatientID"})
    # **Filter valid patients only (avoid NOTES/N/A)**
    df_sh = df_sh[df_sh["PatientID"].astype(str).str.startswith("STS_")]
    clin_parts.append(df_sh)

if clin_parts:
    clin = clin_parts[0]
    for add in clin_parts[1:]:
        clin = clin.merge(add, on="PatientID", how="outer", suffixes=("","_dup"))
    dup_cols = [c for c in clin.columns if c.endswith("_dup")]
    clin = clin.drop(columns=dup_cols)
else:
    clin = pd.DataFrame(columns=["PatientID"])

# ---------------------------
# 7) Build clinical encodings (Cox + DeepSurv)
# ---------------------------
clin_base = clin.drop_duplicates("PatientID").set_index("PatientID")

label_regex_time = r"(time|survival|months|days|follow)"
label_regex_event = r"(event|status|censor|death|progress)"
time_candidates = [c for c in clin_base.columns if re.search(label_regex_time, c, re.I)]
event_candidates = [c for c in clin_base.columns if re.search(label_regex_event, c, re.I)]
label_cols = list(set(time_candidates + event_candidates))

feat_df = clin_base.drop(columns=[c for c in label_cols if c in clin_base.columns], errors="ignore")
numeric_feats = feat_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_feats = [c for c in feat_df.columns if c not in numeric_feats]

feat_num = feat_df[numeric_feats].copy().fillna(feat_df[numeric_feats].median())
feat_cat = feat_df[categorical_feats].copy()

from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Cox + LASSO: one-hot + z-score
cox_num = pd.DataFrame(StandardScaler().fit_transform(feat_num),
                       index=feat_num.index, columns=feat_num.columns)
cox_cat = pd.get_dummies(feat_cat.astype("category"), dummy_na=True)
X_cox = pd.concat([cox_num, cox_cat], axis=1)
X_cox.to_csv(OUTDIR / "clinical_features_cox_lasso.csv")

# DeepSurv: ordinal + z-score
ds_num = pd.DataFrame(StandardScaler().fit_transform(feat_num),
                      index=feat_num.index, columns=feat_num.columns)
if not feat_cat.empty:
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    ds_cat_vals = enc.fit_transform(feat_cat.fillna("NaN").astype(str).values)
    ds_cat = pd.DataFrame(ds_cat_vals, index=feat_cat.index,
                          columns=[f"{c}_ord" for c in feat_cat.columns])
else:
    ds_cat = pd.DataFrame(index=feat_num.index)
X_deepsurv = pd.concat([ds_num, ds_cat], axis=1)
X_deepsurv.to_csv(OUTDIR / "clinical_features_deepsurv.csv")

# ---------------------------
# 8) Derive survival labels (OS / PFS) from clinical columns
# ---------------------------
def pick_exact_or_contains(colnames, target_exact, target_contains):
    for c in colnames:
        if str(c).strip().lower() == target_exact:
            return c
    for c in colnames:
        if target_contains in str(c).strip().lower():
            return c
    return None

full_tmp = rad2.merge(clin, on="PatientID", how="left")

col_time_dx_to_outcome = pick_exact_or_contains(
    full_tmp.columns, "time – diagnosis to outcome (days)", "diagnosis to outcome"
)
col_time_dx_to_fu = pick_exact_or_contains(
    full_tmp.columns, "time – diagnosis to last follow-up (days)", "diagnosis to last follow-up"
)
col_status = pick_exact_or_contains(
    full_tmp.columns, "status (ned, awd, d)", "status"
)

label_cols_found = [c for c in [col_time_dx_to_outcome, col_time_dx_to_fu, col_status] if c is not None]

pat_labels = (full_tmp[["PatientID"] + label_cols_found]
              .drop_duplicates(subset=["PatientID"])
              .set_index("PatientID"))

def norm_status(s):
    if pd.isna(s): return None
    return str(s).strip().upper().replace(" ", "")

def to_float(x):
    try: return float(x)
    except: return np.nan

if col_status:
    pat_labels["Status_norm"] = pat_labels[col_status].apply(norm_status)
else:
    pat_labels["Status_norm"] = None

pat_labels["t_outcome"] = pat_labels[col_time_dx_to_outcome].apply(to_float) if col_time_dx_to_outcome else np.nan
pat_labels["t_fu"]      = pat_labels[col_time_dx_to_fu].apply(to_float) if col_time_dx_to_fu else np.nan

# OS
pat_labels["OS_event"] = (pat_labels["Status_norm"] == "D").astype(int)
pat_labels["OS_time_days"] = np.where(pat_labels["OS_event"]==1, pat_labels["t_outcome"], pat_labels["t_fu"])

# PFS
pat_labels["PFS_event"] = pat_labels["Status_norm"].isin(["AWD","D"]).astype(int)
pat_labels["PFS_time_days"] = np.where(pat_labels["PFS_event"]==1, pat_labels["t_outcome"], pat_labels["t_fu"])

pat_labels[["Status_norm","OS_event","OS_time_days","PFS_event","PFS_time_days"]].to_csv(
    OUTDIR / "survival_labels_by_patient.csv"
)

# ---------------------------
# 9) Save split datasets (+ clinical + labels)
# ---------------------------
def save_split(name):
    df = rad2[rad2["split"]==name].copy()
    df = df.merge(clin, on="PatientID", how="left")
    df = df.merge(pat_labels[["OS_event","OS_time_days","PFS_event","PFS_time_days"]],
                  left_on="PatientID", right_index=True, how="left")
    df.to_csv(OUTDIR / f"{name}_studies.csv", index=False)
    df.to_csv(OUTDIR / f"{name}_studies_labeled.csv", index=False)
    return df

train_df = save_split("train")
val_df   = save_split("val")
test_df  = save_split("test")

# ---------------------------
# 10) Quick summary
# ---------------------------
summary = pd.DataFrame({
    "split": ["train","val","test"],
    "target_unique_studies": [targets["train"], targets["val"], targets["test"]],
    "actual_unique_studies": [
        train_df["StudyID"].nunique(),
        val_df["StudyID"].nunique(),
        test_df["StudyID"].nunique(),
    ],
    "patients": [
        train_df["PatientID"].nunique(),
        val_df["PatientID"].nunique(),
        test_df["PatientID"].nunique(),
    ],
})
print("=== Split summary ===")
print(summary.to_string(index=False))

print("\nOutputs in:", str(OUTDIR))
for f in OUTDIR.iterdir():
    print(" -", f.name)

# Sanity on labels and row counts
print("\n=== Labels summary (per patient) ===")
print("Patients:", pat_labels.shape[0])
print("OS events:", int(pat_labels['OS_event'].sum()),
      "| PFS events:", int(pat_labels['PFS_event'].sum()))
print("Missing OS_time_days:", int(pd.isna(pat_labels['OS_time_days']).sum()),
      "| Missing PFS_time_days:", int(pd.isna(pat_labels['PFS_time_days']).sum()))
print("\nClinical features shapes:",
      "COX", pd.read_csv(OUTDIR/'clinical_features_cox_lasso.csv', index_col=0).shape,
      "| DeepSurv", pd.read_csv(OUTDIR/'clinical_features_deepsurv.csv', index_col=0).shape)
